In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target

# Apply same cleaning as before
df = df[df['MedHouseVal'] < 5.0]
df['rooms_per_person'] = df['AveRooms'] / df['AveOccup']
df['bedrooms_ratio'] = df['AveBedrms'] / df['AveRooms']
df['income_per_occupant'] = df['MedInc'] / df['AveOccup']

print("Data loaded. Shape:", df.shape)

Data loaded. Shape: (19648, 12)


In [4]:
# Feature 1, 2, 3: Distances to major cities
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

SF = (37.7749, -122.4194)
LA = (34.0522, -118.2437)
SD = (32.7157, -117.1611)

df['dist_to_sf'] = haversine(df['Latitude'], df['Longitude'], SF[0], SF[1])
df['dist_to_la'] = haversine(df['Latitude'], df['Longitude'], LA[0], LA[1])
df['dist_to_sd'] = haversine(df['Latitude'], df['Longitude'], SD[0], SD[1])
df['dist_to_nearest_city'] = df[['dist_to_sf', 'dist_to_la', 'dist_to_sd']].min(axis=1)

# Feature 4: Coastal proximity
# California coastline reference points
coastline_points = [
    (41.7, -124.2), (40.5, -124.4), (39.0, -123.7),
    (38.3, -123.0), (37.8, -122.5), (37.2, -122.4),
    (36.6, -121.9), (35.7, -121.3), (34.4, -120.5),
    (34.0, -119.7), (33.7, -118.3), (33.2, -117.4),
    (32.7, -117.2)
]

def dist_to_coast(lat, lon):
    return min(haversine(lat, lon, clat, clon) for clat, clon in coastline_points)

df['coastal_proximity_km'] = df.apply(
    lambda row: dist_to_coast(row['Latitude'], row['Longitude']), axis=1
)

# Feature 5: Income inequality proxy
# High income but high occupancy = wealth concentrated, overcrowded
df['income_inequality_proxy'] = df['MedInc'] / (df['AveOccup'] * df['AveBedrms'])

# Feature 6: Housing density
df['housing_density'] = df['Population'] / df['AveRooms']

print("All features added. New shape:", df.shape)
print("\nNew columns:")
print(df[['dist_to_nearest_city', 'coastal_proximity_km', 
          'income_inequality_proxy', 'housing_density']].describe().round(2))

All features added. New shape: (19648, 19)

New columns:
       dist_to_nearest_city  coastal_proximity_km  income_inequality_proxy  \
count              19648.00              19648.00                 19648.00   
mean                  79.33                 68.19                     1.25   
std                   82.09                 57.10                     0.63   
min                    0.42                  0.00                     0.01   
25%                   19.62                 29.99                     0.77   
50%                   47.02                 46.27                     1.18   
75%                  115.58                 95.15                     1.64   
max                  489.12                341.93                     8.68   

       housing_density  
count         19648.00  
mean            293.19  
std             249.61  
min               0.27  
25%             148.72  
50%             229.90  
75%             358.26  
max            6770.14  


In [5]:
from sklearn.model_selection import train_test_split, cross_val_score
from xgboost import XGBRegressor

X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_v2 = XGBRegressor(n_estimators=100, random_state=42)
scores = cross_val_score(model_v2, X_train, y_train, cv=5, scoring='r2')

print(f"V2 Model R2: {scores.mean():.4f} (+/- {scores.std():.4f})")
print(f"Original model R2 was: 0.8068")
print(f"Improvement: {scores.mean() - 0.8068:.4f}")

V2 Model R2: 0.8208 (+/- 0.0099)
Original model R2 was: 0.8068
Improvement: 0.0140
